# **Transformer Model for Text Classification**

## Text classification
Let's build a text classification model using PyTorch and torchtext to classify news articles into one of the four categories: World, Sports, Business, and Sci/Tech.


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import AG_NEWS

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Import bank dataset**

Load the AG_NEWS dataset for the train split and split it into input text and corresponding labels:


In [2]:
train_iter = AG_NEWS(split = "train")
train_iter

ShardingFilterIterDataPipe

In [3]:
label, text = next(iter(train_iter))
print(f"label:{label}, text: {text}")

label:3, text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [4]:
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
ag_news_label[label]

'Business'

Finding number of class in the dataset

In [5]:
num_classes = len(set([label for (label, text) in train_iter]))
print(f"number of classes: {num_classes}")

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\datapipes\iter\combining.py:297: UserWarning: Some child DataPipes are not exhausted when __iter__ is called. We are resetting the buffer and each child DataPipe will read from the start again.
  warnings.warn("Some child DataPipes are not exhausted when __iter__ is called. We are resetting "


number of classes: 4


### **Build Vocabulary**

In [6]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens (dataset):
    for (label,text) in dataset:
        yield tokenizer(text)

In [7]:
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [8]:
vocab["wall"]

431

#### Build Pipeline to Get Token Indices From Vocabulary

In [9]:
def text_pipeline (input_text):
    return vocab(tokenizer(input_text))

In [10]:
sample_input = "Bears Claw Back Into the Black (Reuters) Reuters"
text_pipeline(sample_input)

[1605, 14838, 113, 66, 2, 848, 13, 27, 14, 27]

## **Dataset Initializatioin**

In [13]:
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import random_split

In [14]:
# initializing training and testing iter
train_iter, test_iter = AG_NEWS()

# initializing training and testing dataset
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

num_train = int(len(train_dataset) * 0.95)

train_split, valid_split = random_split(train_dataset,[num_train,(len(train_dataset)-num_train)])

### **Initializing Device**

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')